# Análise de Criminalidade em Grandes Cidades Brasileiras
**Projeto G2 — Tema 15 | Linguagem de Programação**

Análise exploratória de dados de criminalidade no Brasil entre 2015 e 2024,
utilizando Python, Pandas, Matplotlib, Seaborn e SQLAlchemy.

## 1. Introdução ao Problema

A criminalidade urbana é um dos principais desafios da segurança pública brasileira.
Grandes centros sofrem com altos índices de violência que impactam diretamente a qualidade de vida
da população, a economia local e a capacidade do Estado de prover serviços básicos.

Este notebook realiza uma análise exploratória completa sobre ocorrências, vítimas, prisões e
índices de violência em grandes cidades do Brasil entre 2015 e 2024. O objetivo é identificar
padrões, tendências temporais e fatores associados à criminalidade — como desigualdade de renda
e distribuição regional — para embasar decisões estratégicas de segurança pública com evidências
concretas.

**Perguntas que guiam a análise:**
- Quais regiões e cidades concentram mais crimes?
- Há sazonalidade — meses ou períodos do dia mais críticos?
- Existe relação entre renda média e índice de violência?
- A criminalidade aumentou ou diminuiu ao longo do período?

## 2. Explicação da Base de Dados

A base utilizada é um dataset simulado (`simulacao_criminalidade_brasil.csv`) construído para
representar o cenário de criminalidade em grandes municípios brasileiros de 2015 a 2024.
Os dados foram gerados com distribuições estatísticas compatíveis com fontes públicas como
o Atlas da Violência (IPEA) e o Anuário Brasileiro de Segurança Pública (FBSP).

**Estrutura do dataset:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `data` | datetime | Data do registro (dia/mês/ano) |
| `ano` / `mes` | int | Ano e mês extraídos da data |
| `regiao` | str | Região geográfica do Brasil (Norte, Nordeste, etc.) |
| `uf` | str | Unidade Federativa |
| `cidade` | str | Município |
| `bairro` | str | Bairro do registro |
| `tipo_crime` | str | Categoria do crime (Roubo, Furto, Homicídio, etc.) |
| `periodo_dia` | str | Turno em que o crime ocorreu (Madrugada, Manhã, Tarde, Noite) |
| `ocorrencias` | int | Número de ocorrências registradas |
| `vitimas` | int | Número de vítimas no período |
| `prisoes` | int | Número de prisões efetuadas |
| `renda_media` | float | Renda média mensal do bairro/região (R$) |
| `indice_violencia` | float | Índice composto de violência (0–100) |
| `nivel_risco` | str | Classificação de risco: Baixo, Médio, Alto ou Crítico |

**Cobertura:** 37 municípios distribuídos pelas 5 regiões do Brasil, com registros mensais
de janeiro de 2015 a dezembro de 2024 (120 meses).

## 3. Importação das Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import sqlalchemy as sa
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "#0b0e17"
plt.rcParams["axes.facecolor"] = "#111520"
plt.rcParams["text.color"] = "#e2e8f0"
plt.rcParams["axes.labelcolor"] = "#8892a4"
plt.rcParams["xtick.color"] = "#8892a4"
plt.rcParams["ytick.color"] = "#8892a4"
plt.rcParams["axes.edgecolor"] = "#1e2535"
plt.rcParams["grid.color"] = "#1e2535"

ROXO    = "#7c3aed"
ROXO_LT = "#a78bfa"
VERMELHO = "#ef4444"
VERDE   = "#4ade80"
CORES   = ["#7c3aed","#a78bfa","#6d28d9","#ef4444","#fbbf24","#4ade80"]

print("✅ Bibliotecas importadas com sucesso")

## 4. Leitura dos Dados

In [ ]:
df = pd.read_csv("dados/simulacao_criminalidade_brasil.csv", encoding="utf-8-sig")
df["data"] = pd.to_datetime(df["data"])

print(f"Shape: {df.shape}")
print(f"Período: {df['data'].min().date()} → {df['data'].max().date()}")
print(f"Colunas: {list(df.columns)}")
df.head()

## 5. Limpeza e Preparação dos Dados

In [ ]:
# Verificação de valores nulos — essencial antes de qualquer análise
print("Valores nulos por coluna:")
print(df.isnull().sum())
print()
print("Tipos de dados:")
print(df.dtypes)

In [ ]:
# Verificação de duplicatas
duplicatas = df.duplicated().sum()
print(f"Linhas duplicadas: {duplicatas}")

# Garantir tipos corretos nas colunas numéricas
df["ano"]        = df["ano"].astype(int)
df["mes"]        = df["mes"].astype(int)
df["ocorrencias"] = df["ocorrencias"].astype(int)
df["vitimas"]    = df["vitimas"].astype(int)
df["prisoes"]    = df["prisoes"].astype(int)

# Padronização de strings: remove espaços extras que poderiam quebrar agrupamentos
for col in ["cidade", "uf", "regiao", "tipo_crime", "periodo_dia", "nivel_risco", "bairro"]:
    df[col] = df[col].str.strip()

print("\n✅ Padronização de strings concluída")

In [ ]:
# Verificação de consistência: prisões não podem ser maiores que ocorrências
inconsistencias = df[df["prisoes"] > df["ocorrencias"]]
print(f"Registros com prisões > ocorrências: {len(inconsistencias)}")

# Verificação de valores negativos em colunas que só deveriam ter positivos
for col in ["ocorrencias", "vitimas", "prisoes", "renda_media", "indice_violencia"]:
    negativos = (df[col] < 0).sum()
    print(f"  {col}: {negativos} valores negativos")

# Verificação do intervalo do índice de violência (esperado: 0 a 100)
print(f"\nIndice de violência — min: {df['indice_violencia'].min():.1f} | max: {df['indice_violencia'].max():.1f}")

print("\n✅ Dados prontos para análise")

## 6. Engenharia de Atributos

In [ ]:
# Taxa de prisão por ocorrência — mede a efetividade policial
# replace(0, np.nan) evita divisão por zero
df["taxa_prisao"] = (df["prisoes"] / df["ocorrencias"].replace(0, np.nan)).round(3)

# Faixa de renda categorizada para análise socioeconômica
df["faixa_renda"] = pd.cut(
    df["renda_media"],
    bins=[0, 2000, 3500, 5000, 99999],
    labels=["Baixa", "Média", "Média-Alta", "Alta"]
)

# Semestre — agrupa meses em dois períodos para análise de sazonalidade
df["semestre"] = df["mes"].apply(lambda m: "1º Sem" if m <= 6 else "2º Sem")

print("✅ Novas colunas criadas:")
print(["taxa_prisao", "faixa_renda", "semestre"])
df[["taxa_prisao", "faixa_renda", "semestre"]].head()

## 7. Análise Exploratória

Antes de partir para os gráficos finais, exploramos a distribuição das variáveis,
suas estatísticas descritivas e as correlações entre elas. Isso ajuda a entender
o comportamento dos dados e a embasar as interpretações posteriores.

In [ ]:
# Estatísticas descritivas das colunas numéricas
# Média, desvio padrão, quartis e extremos de uma vez só
cols_num = ["ocorrencias", "vitimas", "prisoes", "renda_media", "indice_violencia", "taxa_prisao"]
df[cols_num].describe().round(2)

In [ ]:
# Distribuição das variáveis categóricas — quantos registros por categoria
print("Registros por região:")
print(df["regiao"].value_counts(), "\n")

print("Registros por tipo de crime:")
print(df["tipo_crime"].value_counts(), "\n")

print("Registros por nível de risco:")
print(df["nivel_risco"].value_counts(), "\n")

print("Registros por faixa de renda:")
print(df["faixa_renda"].value_counts())

In [ ]:
# Matriz de correlação entre variáveis numéricas
# Ajuda a identificar quais variáveis andam juntas e com que intensidade
corr_matrix = df[cols_num].corr().round(2)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(
    corr_matrix,
    annot=True, fmt=".2f", cmap="RdPu",
    linewidths=0.5, linecolor="#0b0e17",
    ax=ax, cbar=False,
    annot_kws={"size": 11, "color": "#e2e8f0"}
)
ax.set_title("Matriz de Correlação — Variáveis Numéricas", color="#e2e8f0", fontsize=14, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot do índice de violência por região
# Mostra a dispersão e os outliers de cada região de uma vez
fig, ax = plt.subplots(figsize=(11, 5))
regioes_ord = (df.groupby("regiao")["indice_violencia"]
               .median().sort_values(ascending=False).index.tolist())
sns.boxplot(
    data=df, x="regiao", y="indice_violencia",
    order=regioes_ord, palette="RdPu", ax=ax
)
ax.set_title("Distribuição do Índice de Violência por Região", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("Índice de Violência")
plt.tight_layout()
plt.show()

In [ ]:
# Ocorrências médias por semestre — existe sazonalidade entre 1º e 2º semestre?
sem = df.groupby(["ano", "semestre"])["ocorrencias"].sum().reset_index()
sem_pivot = sem.pivot(index="ano", columns="semestre", values="ocorrencias")

print("Total de ocorrências por semestre e ano:")
print(sem_pivot.to_string())
print(f"\nMédia 1º Sem: {sem_pivot['1º Sem'].mean():.0f} | Média 2º Sem: {sem_pivot['2º Sem'].mean():.0f}")

## 8. KPIs Principais

In [ ]:
total_oc    = df["ocorrencias"].sum()
total_vit   = df["vitimas"].sum()
total_pris  = df["prisoes"].sum()
idx_medio   = round(df["indice_violencia"].mean(), 1)
cidade_crit = df.groupby("cidade")["ocorrencias"].sum().idxmax()
crime_freq  = df.groupby("tipo_crime")["ocorrencias"].sum().idxmax()
regiao_crit = df.groupby("regiao")["ocorrencias"].sum().idxmax()
cidades_n   = df["cidade"].nunique()
taxa_resolucao = round(total_pris / total_oc * 100, 1)

kpis = {
    "Total de Ocorrências":      f"{total_oc:,}",
    "Total de Vítimas":          f"{total_vit:,}",
    "Total de Prisões":          f"{total_pris:,}",
    "Taxa de Resolução (%)": f"{taxa_resolucao}%",
    "Índice Médio de Violência": str(idx_medio),
    "Cidade Mais Crítica":       cidade_crit,
    "Crime Mais Frequente":      crime_freq,
    "Região Mais Crítica":       regiao_crit,
    "Cidades Monitoradas":       str(cidades_n),
}

print("=" * 45)
print("   KPIs — CRIMINALIDADE NO BRASIL")
print("=" * 45)
for k, v in kpis.items():
    print(f"  {k:<30} {v}")
print("=" * 45)

## 9. Gráficos

### 9.1 Evolução Anual de Ocorrências

In [ ]:
ev = df.groupby("ano")["ocorrencias"].sum().reset_index()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ev["ano"], ev["ocorrencias"], color=ROXO, linewidth=2.5, marker="o",
        markersize=7, markerfacecolor=ROXO_LT)
ax.fill_between(ev["ano"], ev["ocorrencias"], alpha=0.12, color=ROXO)
ax.set_title("Evolução Anual de Ocorrências", color="#e2e8f0", fontsize=14, pad=12)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xticks(ev["ano"])
plt.tight_layout()
plt.show()

### 9.2 Top 10 Cidades Mais Críticas

In [ ]:
rank = (df.groupby("cidade")["ocorrencias"].sum()
        .reset_index().sort_values("ocorrencias", ascending=False).head(10))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(rank["cidade"], rank["ocorrencias"], color=ROXO)
for bar, val in zip(bars, rank["ocorrencias"]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", color="#fbbf24", fontsize=11)
ax.invert_yaxis()
ax.set_title("Top 10 Cidades — Total de Ocorrências", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 9.3 Ocorrências por Tipo de Crime

In [ ]:
tc = (df.groupby("tipo_crime")["ocorrencias"].sum()
      .reset_index().sort_values("ocorrencias", ascending=False))

fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(tc["tipo_crime"], tc["ocorrencias"], color=CORES[:len(tc)])
for bar, val in zip(bars, tc["ocorrencias"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
            f"{val:,}", ha="center", color="#fbbf24", fontsize=11)
ax.set_title("Ocorrências por Tipo de Crime", color="#e2e8f0", fontsize=14, pad=12)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

### 9.4 Heatmap — Crime × Período do Dia

In [ ]:
hc = df.groupby(["tipo_crime", "periodo_dia"])["ocorrencias"].sum().reset_index()
hc_piv = hc.pivot(index="tipo_crime", columns="periodo_dia", values="ocorrencias").fillna(0)
ordem = [p for p in ["Madrugada", "Manhã", "Tarde", "Noite"] if p in hc_piv.columns]
hc_piv = hc_piv[ordem]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(hc_piv, annot=True, fmt=".0f", cmap="RdPu",
            linewidths=0.5, linecolor="#0b0e17",
            ax=ax, cbar=False,
            annot_kws={"size": 11, "color": "#e2e8f0"})
ax.set_title("Heatmap — Tipo de Crime × Período do Dia", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 9.5 Dispersão — Renda Média × Índice de Violência

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    df["renda_media"], df["indice_violencia"],
    c=df["ocorrencias"], cmap="RdPu",
    alpha=0.4, s=15, edgecolors="none"
)
# Linha de tendência linear por regressão de grau 1
m, b = np.polyfit(df["renda_media"], df["indice_violencia"], 1)
xs = np.linspace(df["renda_media"].min(), df["renda_media"].max(), 200)
ax.plot(xs, m * xs + b, color=ROXO_LT, linewidth=1.8, linestyle="--", label="Tendência")
ax.set_title("Renda Média × Índice de Violência", color="#e2e8f0", fontsize=14, pad=12)
ax.set_xlabel("Renda Média (R$)")
ax.set_ylabel("Índice de Violência")
ax.legend()
plt.tight_layout()
plt.show()

corr = df[["renda_media", "indice_violencia"]].corr().iloc[0, 1]
print(f"Correlação de Pearson: {corr:.4f}")

### 9.6 Ocorrências por Região

In [ ]:
reg = (df.groupby("regiao")["ocorrencias"].sum()
       .reset_index().sort_values("ocorrencias", ascending=False))

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(reg["regiao"], reg["ocorrencias"],
              color=[ROXO, ROXO_LT, "#6d28d9", VERMELHO, "#fbbf24"][:len(reg)])
for bar, val in zip(bars, reg["ocorrencias"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{val:,}", ha="center", color="#fbbf24", fontsize=11)
ax.set_title("Ocorrências por Região", color="#e2e8f0", fontsize=14, pad=12)
plt.tight_layout()
plt.show()

## 10. Persistência com SQLAlchemy e SQLite

In [ ]:
engine = sa.create_engine("sqlite:///database/criminalidade.db", echo=False)

# Persiste o DataFrame principal
df.to_sql("ocorrencias", engine, if_exists="replace", index=False)
print("✅ Tabela 'ocorrencias' gravada com sucesso")

# Tabela de KPIs
kpis_df = pd.DataFrame(list(kpis.items()), columns=["indicador", "valor"])
kpis_df.to_sql("kpis", engine, if_exists="replace", index=False)
print("✅ Tabela 'kpis' gravada com sucesso")

In [ ]:
# Consulta de exemplo com SQL — confirma que os dados foram gravados corretamente
with engine.connect() as conn:
    resultado = pd.read_sql(
        "SELECT cidade, SUM(ocorrencias) as total "
        "FROM ocorrencias GROUP BY cidade "
        "ORDER BY total DESC LIMIT 10",
        conn
    )
print("Top 10 cidades via SQL:")
resultado

## 11. Interpretação dos Resultados

- **Distribuição regional:** A região Sudeste concentra o maior volume absoluto de ocorrências,
reflexo da sua densidade populacional. Entretanto, regiões como Norte e Nordeste apresentam
índices de violência médios mais elevados proporcionalmente, o que sugere maior vulnerabilidade
per capita mesmo com volume menor de registros.

- **Tipos de crime:** Roubo e furto lideram o ranking de frequência. Crimes como homicídio,
embora menos frequentes, apresentam maior índice de violência associado — o que indica gravidade
desproporcional ao volume.

- **Horários críticos:** O período noturno concentra crimes patrimoniais, enquanto violência
doméstica se distribui de forma mais uniforme ao longo do dia. A madrugada aparece como
o período de menor volume mas maior gravidade por ocorrência.

- **Renda e violência:** A correlação de Pearson entre renda média e índice de violência
confirma uma relação inversamente proporcional — regiões com menor renda tendem a apresentar
indicadores mais elevados. As exceções observadas no scatter indicam a influência de outros
fatores, como presença policial e densidade urbana.

- **Tendência temporal:** A série histórica revela variações anuais associadas a ciclos
econômicos e implementação de políticas de segurança pública. A análise semestral não
evidenciou sazonalidade marcante, mas há variação por tipo de crime.

- **Taxa de resolução:** A relação prisões/ocorrências indica espaço para melhoria na
efetividade operacional em algumas modalidades criminais específicas.

## 12. Conclusão

Este projeto demonstrou que a análise de dados de criminalidade permite identificar padrões
relevantes para o planejamento de políticas públicas de segurança. Os principais achados são:

1. **A criminalidade não é uniforme** — há concentração clara em cidades, regiões e períodos
específicos, o que permite direcionar recursos de forma mais eficiente.

2. **Existe correlação negativa entre renda média e índice de violência**, reforçando que
políticas de redução da desigualdade têm impacto direto na segurança pública.

3. **O período noturno é criticamente mais perigoso para crimes patrimoniais**, sugerindo
que policiamento ostensivo noturno teria maior retorno em termos de prevenção.

4. **A tendência histórica permite projeções e planejamento preventivo**, especialmente
para cidades que apresentam crescimento consistente nos indicadores ao longo dos anos.

O dashboard interativo desenvolvido em Streamlit complementa esta análise com filtros
dinâmicos e visualizações interativas, tornando os insights acessíveis a tomadores de
decisão que não necessariamente têm familiaridade com código.